# Module 03: Neural Networks & Deep Learning

**Course:** Introduction to Modern AI  
**Estimated Time:** 90 minutes  
**Prerequisites:** Module 01, Module 02

---

## 1. Why Deep Networks?

Linear models cannot classify data that isn't separable by a straight line. By stacking multiple layers of neurons with non-linear activation functions (like `ReLU`), neural networks can learn complex, curved boundaries.

In [ ]:
# Install PyTorch and dependencies
!pip install -q torch torchvision scikit-learn numpy

## 2. Building a PyTorch Classifier for Non-Linear Data

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Generate Non-Linear Data (Concentric Circles)
X, y = make_circles(n_samples=1000, noise=0.05, factor=0.5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_test_t  = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# 2. Define Network Architecture
class CircleClassifier(nn.Module):
    def __init__(self):
        super(CircleClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        return self.net(x)

model = CircleClassifier()
loss_fn = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.03)

# 3. Training Loop
epochs = 200
print("--- TRAINING NEURAL NETWORK ---")
for epoch in range(1, epochs + 1):
    model.train()
    y_pred = model(X_train_t)
    loss = loss_fn(y_pred, y_train_t)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 50 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            test_preds = (model(X_test_t) >= 0.5).float()
            acc = accuracy_score(y_test, test_preds.numpy())
        print(f"Epoch {epoch:3d} | Train Loss: {loss.item():.4f} | Test Accuracy: {acc * 100:.1f}%")

print(f"\nFinal Test Accuracy: {acc * 100:.2f}%")